## PILLAR 6: NETWORKING

### Network Interface

A **network interface** is the OS's abstraction for a point of network connectivity — think of it as a named "network endpoint" the kernel exposes, whether or not there's literal hardware behind it. `eth0`, `lo`, `wlan0`, `docker0` are all interfaces. Each one has: a name, a MAC address (if physical/virtual hardware-backed), one or more IP addresses assigned to it, and a **state**.

**"Up" vs "down" is literally an on/off switch for that interface**, independent of whether it has a valid IP or a cable plugged in:

- **UP** — the interface is administratively enabled; the kernel will attempt to send/receive traffic through it.
- **DOWN** — the interface is administratively disabled; even if it's got a perfectly valid IP configured and a cable plugged in, _nothing_ goes in or out. It's like a Django app being in your codebase but `INSTALLED_APPS` doesn't include it — the config exists, but it's not participating.

You'll notice the example output actually shows **two separate flags** that both matter:

In [ ]:
<BROADCAST,MULTICAST,UP,LOWER_UP>

**Developer bridge:** this is like the difference between a Django URL pattern existing in `urls.py` (administratively "up" — the route is registered) versus the actual upstream service it proxies to being reachable (`LOWER_UP` — the underlying connection is live). You can have one without the other: a route registered but backend down (`UP` but not `LOWER_UP`), or code removed but server still running (`LOWER_UP` but not `UP`, less common but analogous).

**Why this distinction matters in support:** if a ticket says "the server lost network," `UP` but no `LOWER_UP` tells you it's a physical/link problem (cable, driver, virtual switch) — the OS side is configured correctly and ready, but nothing is connected. `DOWN` entirely tells you someone (or some script/reboot state) disabled the interface itself — a config/administrative issue, not a physical one. Two completely different fix paths from the exact same symptom.

---

First up: `ip addr` — the modern replacement for `ifconfig`.

**COMMAND:** `ip addr`

**PURPOSE (support):** Almost every network ticket starts the same way: "the server can't reach X" or "I can't connect to this machine." Before you can diagnose DNS, routing, or firewall issues, you need the fundamentals:
- **what IP address does this machine actually have**
- **is the interface even up**
- **is it the interface you think it is**.
> **Note**: `ip addr` is step zero for **any network ticket**.

**SYNTAX:** `ip addr [show] [dev INTERFACE]` (often abbreviated `ip a`)

**FLAGS:**

|Flag/Form|Meaning|
|---|---|
|`ip addr` / `ip a`|show all interfaces and their addresses|
|`ip addr show dev eth0`|show just one interface|
|`ip -4 addr`|IPv4 only|
|`ip -6 addr`|IPv6 only|
|`ip -br addr`|brief, one-line-per-interface summary|

**Developer bridge:** think of this like `docker network inspect` but for the host itself — it tells you which "network" (interface) exists, what IP it got, and whether it's actually attached/up, the same categories of info you'd check when a container can't reach another container.

**EXAMPLE:**

In [ ]:
$ ip addr
1: lo: <LOOPBACK,UP,LOWER_UP> mtu 65536 qdisc noqueue state UNKNOWN group default qlen 1000
    link/loopback 00:00:00:00:00:00 brd 00:00:00:00:00:00
    inet 127.0.0.1/8 scope host lo
       valid_lft forever preferred_lft forever
    inet6 ::1/128 scope host
       valid_lft forever preferred_lft forever
2: eth0: <BROADCAST,MULTICAST,UP,LOWER_UP> mtu 1500 qdisc mq state UP group default qlen 1000
    link/ether 00:15:5d:c4:9a:1f brd ff:ff:ff:ff:ff:ff
    inet 172.20.144.12/20 brd 172.20.159.255 scope global eth0
       valid_lft forever preferred_lft forever
    inet6 fe80::215:5dff:fec4:9a1f/64 scope link
       valid_lft forever preferred_lft forever

**READ THE OUTPUT:**

- **`1: lo`** — the loopback interface, always `127.0.0.1`, always present. This is "talking to yourself" (`localhost`).
- **`2: eth0`** — your real network interface. `<UP,LOWER_UP>` means the interface is administratively up _and_ has a physical/virtual link (cable/connection actually active) — both need to be true for real connectivity.
- **`inet 172.20.144.12/20`** — the actual IP address and its subnet mask in CIDR notation (`/20`). This is what other machines use to reach this one.
- **`link/ether 00:15:5d:...`** — the MAC address, hardware-level identifier.
> **State `UP` vs `DOWN`** — if this says `DOWN`, nothing will work regardless of IP config — that's your first checkpoint, before you even look at the IP.

**⚠️ WSL FLAG:** Your `eth0` IP here (`172.20.144.12`) is **not** the same as your Windows host's IP on your real network. WSL2 creates a virtual NAT'd network — this address is only reachable from inside WSL and from the Windows host via that NAT, not from other devices on your LAN directly. If a ticket scenario ever involves "can another machine reach this WSL box," that's a fundamentally different (and more complex) question than what `ip addr` alone answers — flagged now, we'll cover it properly when we hit the WSL networking section later in this pillar.

**TRY NOW:**

In [ ]:
dennie@HP:~/linux-tut$ ip addr
1: lo: <LOOPBACK,UP,LOWER_UP> mtu 65536 qdisc noqueue state UNKNOWN group default qlen 1000
    link/loopback 00:00:00:00:00:00 brd 00:00:00:00:00:00
    inet 127.0.0.1/8 scope host lo
       valid_lft forever preferred_lft forever
    inet6 ::1/128 scope host
       valid_lft forever preferred_lft forever
2: eth0: <BROADCAST,MULTICAST,UP,LOWER_UP> mtu 1500 qdisc mq state UP group default qlen 1000
    link/ether 00:15:5d:5c:01:d4 brd ff:ff:ff:ff:ff:ff
    inet 172.26.126.1/20 brd 172.26.127.255 scope global eth0
       valid_lft forever preferred_lft forever
    inet6 fe80::215:5dff:fe5c:1d4/64 scope link
       valid_lft forever preferred_lft forever

In [ ]:
dennie@HP:~/linux-tut$ ip -br addr
lo               UNKNOWN        127.0.0.1/8 ::1/128
eth0             UP             172.26.126.1/20 fe80::215:5dff:fe5c:1d4/64

In [ ]:
dennie@HP:~/linux-tut$ ip addr show dev eth0
2: eth0: <BROADCAST,MULTICAST,UP,LOWER_UP> mtu 1500 qdisc mq state UP group default qlen 1000
    link/ether 00:15:5d:5c:01:d4 brd ff:ff:ff:ff:ff:ff
    inet 172.26.126.1/20 brd 172.26.127.255 scope global eth0
       valid_lft forever preferred_lft forever
    inet6 fe80::215:5dff:fe5c:1d4/64 scope link
       valid_lft forever preferred_lft forever

All three ran clean and match what we'd expect:

- **`ip addr`** — full output shows `eth0` at `172.26.126.1/20`, both `UP` and `LOWER_UP` present, exactly what healthy connectivity looks like.
- **`ip -br addr`** — same info, condensed to one line per interface. See how much faster that is to eyeball during an incident: `eth0 UP 172.26.126.1/20` tells you everything you need in one glance instead of parsing 6 lines.
- **`ip addr show dev eth0`** — scoped to just the one interface, useful once you already know which interface you care about and don't want `lo` cluttering the output.

> One real detail worth noting: your IP this session (`172.26.126.1`) is _different_ from the one in my example (`172.20.144.12`) — that's WSL2 reassigning a new NAT address, which commonly happens across `wsl --shutdown`/reboots or even just Windows networking changes. This is actually a live example of the WSL flag from the lesson — don't hardcode or assume a WSL IP stays stable across sessions, always re-check with `ip addr` rather than trusting a cached value from a previous session or runbook.

Compare the full output to the `-br` (brief) version — note how much faster the brief form is to scan when you just need a quick "is this interface up and what's its IP" answer during a live incident.

**TICKET SCENARIO:** "User says their app can't bind to port 8000 — 'address already in use' or 'cannot assign requested address.'" First move here is confirming what interfaces/addresses actually exist on the box — if the app config is trying to bind to a specific IP that doesn't match anything in `ip addr` output, that's the whole bug right there, before you even look at the app.

**GOTCHA:** `ifconfig` still exists on many systems (often needs `net-tools` installed) but is considered legacy/deprecated — don't be surprised if a minimal server doesn't have it at all. `ip` is the standard now; know it as primary, recognize `ifconfig` output if you see it in old runbooks or ticket history.

---

### PowerShell & Windows
#### Public & Privat IPs

In [ ]:
PS C:\Users\denni> Get-NetIPAddress -AddressFamily IPv4 | Select InterfaceAlias, IPAddress, PrefixLength

InterfaceAlias                     IPAddress      PrefixLength
--------------                     ---------      ------------
vEthernet (WSL (Hyper-V firewall)) 172.30.240.1             20
Local Area Connection* 2           169.254.192.33           16
Local Area Connection* 1           169.254.171.37           16
Wi-Fi                              192.168.1.105            24
Ethernet                           169.254.30.231           16
Loopback Pseudo-Interface 1        127.0.0.1                 8

#### Reading the output, row by row

|Interface|IP / prefix|What it is|Verdict|
|---|---|---|---|
|vEthernet (WSL (Hyper-V firewall))|172.30.240.1/20|Windows' end of the virtual link to WSL|Normal, and it's the one that matters here|
|Wi-Fi|192.168.1.105/24|Your real network connection (home router LAN)|Normal, this is your actual internet path|
|Loopback Pseudo-Interface 1|127.0.0.1/8|Windows' `lo`|Normal|
|Local Area Connection* 1 and 2|169.254.x.x/16|Wi-Fi Direct / hotspot virtual adapters|Idle, ignore|
|Ethernet|169.254.30.231/16|Physical NIC with no cable or no DHCP answer|Idle, ignore|

#### The key finding: WSL and Windows share a subnet

- WSL `eth0`: `172.30.240.133/20`
- Windows `vEthernet (WSL)`: `172.30.240.1/20`

A /20 mask means the first 20 bits are the network part. Both addresses start `172.30.240` + the same upper bits, so they're on the same network (172.30.240.0 – 172.30.255.255, which matches the `brd 172.30.255.255` in your selection). So `.1` is Windows and acts as WSL's **default gateway**. That's why WSL can reach the internet: traffic goes WSL → 172.30.240.1 → NAT → Wi-Fi.

#### The 169.254.x.x addresses (APIPA)

`169.254.0.0/16` is the link-local range. A machine self-assigns one when it asks for DHCP and nobody answers.

**WHAT TO LOOK FOR:** In a ticket, a user's _main_ adapter showing `169.254.x.x` means "DHCP failed", so check cable, router, or DHCP server. Here it's on unused adapters, so it's noise. Your real adapter, Wi-Fi, has a proper `192.168.1.105`.

#### Three different address "scopes" in one output

- `127.0.0.1/8`: loopback, never leaves the machine
- `192.168.x.x`, `172.30.x.x`: private RFC1918 addresses, only meaningful inside a LAN
- `169.254.x.x`: link-local, "no DHCP" sign

---

#### Private IP

You already have it from your last output: **192.168.1.105** on Wi-Fi. This command shows only the active adapter:

In [ ]:
PS C:\Users\denni> Get-NetIPAddress -AddressFamily IPv4 -InterfaceAlias "Wi-Fi" | Select IPAddress, PrefixLength

IPAddress     PrefixLength
---------     ------------
192.168.1.105           24

**Private IP** `192.168.1.105/24`. Your PC inside the home LAN (Wi-Fi)

To find the adapter that actually carries your internet traffic, without guessing from the alias:

In [ ]:
PS C:\Users\denni> Get-NetRoute -DestinationPrefix 0.0.0.0/0 | Sort RouteMetric | Select -First 1 InterfaceAlias, NextHop

InterfaceAlias NextHop
-------------- -------
Wi-Fi          192.168.1.1

**Default gateway** `192.168.1.1`. Your router; all internet traffic goes here first.

> `NextHop` is your **router** (the default gateway, likely `192.168.1.1`). `InterfaceAlias` is the adapter that holds your real private IP.

---

#### Public IP
Your PC doesn't know its public IP, because the **router's NAT owns it**. You have to ask an outside service:

In [ ]:
PS C:\Users\denni> (Invoke-RestMethod https://api.ipify.org)
37.188.205.146

**Public IP** `37.188.205.146`. What websites see; shared by every device on your network.

Alternative:

In [ ]:
PS C:\Users\denni> (Invoke-RestMethod https://ifconfig.me/ip)
37.188.205.146

Both services returned the same public IP, so the answer is reliable.

Alternative, https://whatismyipaddress.com/.

#### How to read the two results

- **Private (192.168.1.105):** your address inside the home LAN. It's only meaningful behind your router.
- **Public:** the address websites see. Every device on your network shares it.
- **/24 network:** `192.168.1.0` – `192.168.1.255`, with 254 usable hosts. Your PC (`.105`) and router (`.1`) are on the same network, so they talk directly with no routing.
- **The path out:** PC (192.168.1.105) → router (192.168.1.1) → NAT swaps in 37.188.205.146 → internet.
- **Not CGNAT:** `37.188.205.146` is outside `100.64.0.0/10` and outside the private ranges, so it's a real public address. Port forwarding from your router would work in principle, though some ISPs still block inbound traffic.
- **No VPN sign:** the route goes straight out via Wi-Fi, so the public IP should be your ISP's.

**WHAT TO LOOK FOR:** If the public IP is in `100.64.0.0/10`, your ISP is using CGNAT. You then can't host or port-forward from home, which matters for support tickets about "remote access doesn't work". If you're on a VPN, the public IP will be the VPN's, not your ISP's.

**WHAT TO LOOK FOR:** In a ticket, the private and public IPs answer different questions. Private IP: "can the device reach the router or other LAN devices?" Public IP: "what will the remote server, firewall, or allowlist see?" If a customer says "my IP is allowlisted but still blocked", compare their public IP against the allowlist. Home ISPs often change it.

#### Try now

Confirm the gateway is reachable, then trace the path out:

In [ ]:
Test-Connection 192.168.1.1 -Count 3
Test-NetConnection 8.8.8.8 -TraceRoute

The first hop of the trace should be `192.168.1.1`. Paste the output and we'll read the hops together.

**Theory follow-up:** after practicing, the freeCodeCamp course's "IPv4 Logical Addressing" (48:09) covers private vs public ranges. For NAT, search PowerCert Animated Videos for "NAT explained".